# A real reasoner against the definitions

The Semantic Web reader's version of the walkthrough: the graphs are Turtle files in `examples/`, the practice's side is a deployed reasoner, and the semantics' side is the paper's definitions. Both verdicts are printed side by side.

* **Practice's side.** `owlrl`, an off-the-shelf RDFS / OWL 2 RL reasoner, materializes the graph under the OWL 2 RL/RDF rules (the RDFS profile has no disjointness rule, so OWL 2 RL is used throughout; axiomatic triples switched off). Inconsistency is owlrl's *error* triple. The query is a SPARQL `ASK` whose pattern is `H` with blank nodes as variables, which is simple entailment on standard graphs (Lemma 1). This is "materialize, then query", the operation Theorem 2 says the semantics recovers.
* **Semantics' side.** The functions of `issrdf`, computed from the definitions over an explicit universe, with the two rules that bear on these graphs: rdfs9 and cax-dw. owlrl's regime is far larger (its closure of the three-triple clash graph has 115 triples, 106 of them axiomatic), but on these queries only those two rules matter; the agreement of the *full* rule set with the semantics is what `checks/check_owlrl.py` tests over hundreds of random graphs, with owlrl as the closure on both sides.

The only code in this notebook that is not an import is the conversion of rdflib terms to the short-name convention of `issrdf` (namespace stripped, blank nodes renamed `_:x`, `_:y` in a fixed order): input and output, not a definition. Requires `pip install -r requirements.txt`; the notebook prints a skip line otherwise.

In [1]:
import sys, pathlib; sys.path.insert(0, '..'); sys.path.insert(0, '../checks')
try:
    from rdflib import Graph, URIRef, BNode
    import check_owlrl as W            # close (owlrl materialization), ask (SPARQL ASK), def4_side
    HAVE = True
except ImportError as e:
    HAVE = False; print("skipped: rdflib/owlrl not installed --", e)
from issrdf import (Universe, BOT, make_closure, r_entails, r_inconsistent, make_good,
                    iss_entails, iss_incoherent, content_pos, content_neg, adj, UNIT, show)
from issrdf.show import graph, regime, report, verdict_line
verdicts = []

def short(u, bn, letters='xyzw'):
    '''rdflib term -> issrdf name: namespace stripped; blank nodes renamed _x, _y, ... in order of
    first appearance (G's from 'x', H's from 'y', as in the walkthrough).'''
    if isinstance(u, BNode):
        return bn.setdefault(u, '_' + letters[len(bn)])
    return str(u).split('#')[-1].rstrip('/').split('/')[-1]

def load(path, letters='xyzw'):
    '''A Turtle file as (rdflib triples, issrdf graph).'''
    g = Graph().parse(path, format='turtle'); bn = {}
    G = frozenset(tuple(short(u, bn, letters) for u in t) for t in sorted(g, key=str))
    return frozenset(g), G

## The universe and the regime on the semantics' side

Names from the Turtle files, the three vocabulary IRIs of the two rules, and one spare IRI for Definition 12. The regime is the one used in the walkthrough.

In [2]:
U = Universe(V=('type', 'subClassOf', 'disjointWith'), INDIV=('tweety', 'Bird', 'Flier'), SPARE=('s',))
R = (((('X', 'type', 'Y'), ('Y', 'subClassOf', 'Z')), ('X', 'type', 'Z')),                 # rdfs9
     ((('X', 'type', 'Y'), ('X', 'type', 'Z'), ('Y', 'disjointWith', 'Z')), BOT))           # cax-dw
closure = make_closure(R, U.V); good = make_good(closure)
print(U); print(regime(R))

Universe(V=('type', 'subClassOf', 'disjointWith'), INDIV=('tweety', 'Bird', 'Flier'), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))  # N = ('tweety', 'Bird', 'Flier', 'type', 'subClassOf', 'disjointWith', 's')
{(X, type, Y), (Y, subClassOf, Z)} → (X, type, Z)
{(X, type, Y), (X, type, Z), (Y, disjointWith, Z)} → ⊥


## The two sides, as functions

`owlrl_side` materializes with owlrl and asks; `semantics_side` runs Definition 7 over the contents of Definitions 10–11 against the frame of Definitions 13–14. Both take the graphs in their own representation.

In [3]:
EX = 'http://ex.org/'
def ex_triples(g):
    '''The triples of an rdflib graph over the example namespace, as short names (for display).'''
    bn = {}
    return sorted(tuple(short(u, bn) for u in t) for t in g if any(str(u).startswith(EX) for u in t))

def owlrl_side(rG, rH, label):
    g, inconsistent = W.close(rG, W.OWLRL_Semantics)
    print(f"  owlrl: closure has {len(g)} triples; over ex: {graph(ex_triples(g))}")
    print(f"         inconsistent (error triple): {inconsistent}")
    if rH is not None:
        print(f"         ASK H with blank nodes as variables: {W.ask(g, rH)}")
        return inconsistent or W.ask(g, rH)
    return inconsistent

def semantics_side(G, H, label):
    cl = closure(G)
    print(f"  definitions: cl_R(G) \\ G = {graph(cl - G)}")
    if H is not None:
        F = adj(content_pos(G, U), content_neg(H, U))
        print(f"               [[G]]+ ⊔ [[H]]-: {len(F)} pair(s); first pair and its reason:")
        report(closure, F, limit=1)
        return iss_entails(good, G, H, U)
    F = adj(content_pos(G, U), UNIT)
    print(f"               [[G]]+ ⊔ unit: {len(F)} pair(s):")
    report(closure, F, limit=1)
    return iss_incoherent(good, G, U)

def case(label, g_file, h_file=None, h_turtle=None):
    if not HAVE:
        print(f"{label}: skipped"); return
    rG, G = load(g_file)
    if h_file: rH, H = load(h_file, 'yzw')
    elif h_turtle:
        gh = Graph().parse(data=h_turtle, format='turtle'); bn = {}
        rH, H = frozenset(gh), frozenset(tuple(short(u, bn, 'yzw') for u in t) for t in sorted(gh, key=str))
    else: rH, H = None, None
    assert U.admissible(G, H or frozenset())
    print(f"== {label} ==\n  G = {graph(G)}  ({g_file})")
    if H is not None: print(f"  H = {graph(H)}  ({h_file or 'inline'})")
    a = owlrl_side(rG, rH, label); b = semantics_side(G, H, label)
    line = verdict_line(label, b, a).replace('rules', 'owlrl')
    print(line + '\n'); verdicts.append((label, b, a))

## Cases

1. `tweety.ttl` against `tweety_query.ttl`: is tweety a Flier? rdfs9 derives it; the semantics' single pair is in 𝕀_R.
2. `tweety.ttl` against `tweety_query_bnode.ttl`: is something a Flier? A blank node in `H`; Definition 11's seven instances over `N`.
3. `tweety.ttl` against a query it does not entail: is Flier a tweety?
4. `clash.ttl` alone: owlrl derives *false*; ⟦G⟧ ⊨ ∅ (Proposition 2).
5. `clash.ttl` against the non-entailed query of case 3: an inconsistent graph entails everything, on both sides (explosion; the paragraph after Definition 13).

In [4]:
case("1: tweety, ground query",        '../examples/tweety.ttl', '../examples/tweety_query.ttl')
case("2: tweety, blank-node query",    '../examples/tweety.ttl', '../examples/tweety_query_bnode.ttl')
case("3: tweety, non-entailed query",  '../examples/tweety.ttl',
     h_turtle='@prefix ex: <http://ex.org/> . @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> . ex:Flier rdf:type ex:tweety .')
case("4: clash, incoherent (Prop. 2)", '../examples/clash.ttl')
case("5: clash, non-entailed query",   '../examples/clash.ttl',
     h_turtle='@prefix ex: <http://ex.org/> . @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> . ex:Flier rdf:type ex:tweety .')

== 1: tweety, ground query ==
  G = {(Bird, subClassOf, Flier), (tweety, type, Bird)}  (../examples/tweety.ttl)
  H = {(tweety, type, Flier)}  (../examples/tweety_query.ttl)
  owlrl: closure has 112 triples; over ex: {(Bird, sameAs, Bird), (Bird, subClassOf, Flier), (Flier, sameAs, Flier), (tweety, sameAs, tweety), (tweety, type, Bird), (tweety, type, Flier)}
         inconsistent (error triple): False
         ASK H with blank nodes as variables: True
  definitions: cl_R(G) \ G = {(tweety, type, Flier)}
               [[G]]+ ⊔ [[H]]-: 1 pair(s); first pair and its reason:
  ✓ ⟨{(Bird, subClassOf, Flier), (tweety, type, Bird)}, {(tweety, type, Flier)}⟩
      (tweety, type, Flier) ∈ Δ ∩ cl_R(Γ)  (Definition 13)
1: tweety, ground query                  semantics True   owlrl True   agree

== 2: tweety, blank-node query ==
  G = {(Bird, subClassOf, Flier), (tweety, type, Bird)}  (../examples/tweety.ttl)
  H = {(_:y, type, Flier)}  (../examples/tweety_query_bnode.ttl)
  owlrl: closure has 

## Summary

The reasoner and the definitions agree on every case. The last cell writes the verdicts to `results/owlrl_example.txt`.

In [5]:
if HAVE:
    lines = [verdict_line(label, s, r).replace('rules', 'owlrl') for label, s, r in verdicts]
    print('\n'.join(lines))
    assert [s for _, s, _ in verdicts] == [True, True, False, True, True] and all(s == r for _, s, r in verdicts)
    show.write_log('../results/owlrl_example.txt', lines + [f"summary: {len(verdicts)} cases, 0 disagreements"])
else:
    print("skipped: rdflib/owlrl not installed")

1: tweety, ground query                  semantics True   owlrl True   agree
2: tweety, blank-node query              semantics True   owlrl True   agree
3: tweety, non-entailed query            semantics False  owlrl False  agree
4: clash, incoherent (Prop. 2)           semantics True   owlrl True   agree
5: clash, non-entailed query             semantics True   owlrl True   agree
wrote ../results/owlrl_example.txt (6 lines)
